In [ ]:
%env PYTHONHASHSEED=0
import os
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns
import numpy as np
import anndata as ad
import scanpy as sc
import rpy2
import gseapy
from gseapy import Msigdb

In [ ]:
np.random.seed(0)
sc.set_figure_params(dpi = 300, dpi_save = 300, frameon = False)

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(ggplot2)
library(ggpubr)
library(SingleCellExperiment)
library(dplyr)
library(edgeR)
library(extrafont)
library(ComplexHeatmap)
loadfonts()

In [ ]:
# load astrocytes anndata object
oligos = sc.read_h5ad('../output/human/human_oligodendrocytes_scanpy_object_postclustering.h5ad')

In [ ]:
oligos

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (4, 4), "grid.alpha":0}):
    sc.pl.umap(oligos, color=['final_clusters'], size = 25, frameon=False, title='',
              legend_loc='on data',
           legend_fontsize = 'xx-small',
           legend_fontweight='medium',
           legend_fontoutline=0.75)

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (4, 4), "grid.alpha":0}):
    sc.pl.umap(oligos, color=['final_clusters'], size = 25, frameon=False, title='', legend_loc = None)

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (2, 1), "grid.alpha":0}):
    ax = sc.pl.umap(oligos, color=['final_clusters'], size = 0, frameon=False, title='', show = False)
    ax.legend(loc="upper center", ncol = 10, markerscale=2, fontsize = 15, frameon=False)

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (4, 4), "grid.alpha":0}):
    sc.pl.umap(oligos, color=['sampleid'], size = 25, frameon=False, title='', legend_loc = None)

In [ ]:
# modify sample name annotation to remove underscore
oligos.obs['sampleid_nounderscore'] = (
    oligos.obs["sampleid"]
    .map(lambda x: {
        'Control_A':'Control A',
        'Control_B':'Control B',
        'Control_C':'Control C',
        'CART_A':'CAR T A',
        'CART_B':'CAR T B',
        'CART_C':'CAR T C',
        'CART_D':'CAR T D',
                   }.get(x, x))
    .astype("category")
)

# re-order cell type labels
oligos.obs['sampleid_nounderscore'] = oligos.obs['sampleid_nounderscore'].cat.reorder_categories(['CAR T A',
                                                                                                  'CAR T B',
                                                                                                  'CAR T C',
                                                                                                  'CAR T D',
                                                                                                  'Control A',
                                                                                                  'Control B',
                                                                                                  'Control C'
                                                                                             ])

In [ ]:
# plot UMAP with sample overlaid
with rc_context({"figure.figsize": (4, 4), "grid.alpha":0}):
    sc.pl.umap(oligos, color=['sampleid_nounderscore'], frameon=False, title='', size = 25,
               palette = {"CAR T A":"#662506", "CAR T B":"#cc4c02", "CAR T C":"#fb9a29", "CAR T D":"#fee391", 
                         "Control A":"#2171b5", "Control B":"#6baed6", "Control C":"#bdd7e7"}
              )

In [ ]:
%%R
# plot cell type proportions using propeller functions
library(speckle)

In [ ]:
# extract metadata for creating SCE object
cluster_labels = oligos.obs['final_clusters']
sample_labels = oligos.obs['sampleid_nounderscore']
group_labels = oligos.obs['condition']

In [ ]:
%%R -i cluster_labels -i sample_labels -i group_labels
# create SCE object
sce <- SingleCellExperiment(list(counts=matrix(ncol = length(sample_labels), nrow = 1)),
                     colData=data.frame(clusters=cluster_labels,
                                        sample=sample_labels,
                                        group=group_labels))

In [ ]:
%%R

prop.list <- getTransformedProps(colData(sce)$clusters, sample = colData(sce)$sample, transform = "asin")

In [ ]:
%%R

df = data.frame(Proportions = as.vector(t(prop.list$Proportions)), Samples = rep(colnames(prop.list$Proportions), nrow(prop.list$Proportions)), Clusters = rep(rownames(prop.list$Proportions), each=ncol(prop.list$Proportions)))

In [ ]:
%%R -w 4 -h 3 -r 300 --units in

color_dict <- c('OPC-Control'= 'antiquewhite',
 'OPC-A'= '#9e0142',
 'OPC-B'= '#d8434e',
 'OPC-C'= '#f67a49',
 'OPC-D'= '#fdbf6f',
 'OLG-Control'= 'lightgray',
'OLG-B'= '#bfe5a0',
 'OLG-C'= '#74c7a5',
 'OLG-D'= '#378ebb',
 'COPs'= '#f1f9a9'
)

df$Clusters <- factor(df$Clusters, levels = rev(c('OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D', 'OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D', 'COPs')))

ggplot(df, aes(x = Samples, y = Proportions, fill = Clusters)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = color_dict, guide = NULL) +
  scale_x_discrete(limits = c('Control A', 'Control B', 'Control C', 'CAR T A', 'CAR T B', 'CAR T C', 'CAR T D')) + 
  scale_y_continuous(expand = c(0, 0)) +  # Optional: remove space between bars and x-axis
  theme_minimal() + theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
# perform proportion testing splitting OPCs and OLIGs (we want to test condition enrichment while accounting for differences in OPC vs OLG capture across samples)

In [ ]:
opc = oligos[oligos.obs['final_clusters'].isin(['OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D'])]

In [ ]:
opc

In [ ]:
# extract metadata for creating SCE object
cluster_labels = opc.obs['final_clusters']
sample_labels = opc.obs['sampleid_nounderscore']
group_labels = opc.obs['condition']

In [ ]:
%%R -i cluster_labels -i sample_labels -i group_labels
# create SCE object
sce <- SingleCellExperiment(list(counts=matrix(ncol = length(sample_labels), nrow = 1)),
                     colData=data.frame(clusters=cluster_labels,
                                        sample=sample_labels,
                                        group=group_labels))

In [ ]:
%%R
# Run propeller testing for cell type proportion differences between the two 
# groups
propeller_results <- propeller(clusters = sce$clusters, sample = sce$sample, 
group = sce$group, transform = "asin")

propeller_results

In [ ]:
%%R

saveRDS(propeller_results, "../output/human/human_oligos_OPCs_propeller_differentialabundance_test_results.rds")
write.csv(propeller_results, "../output/human/human_oligos_OPCs_propeller_differentialabundance_test_results.csv")

In [ ]:
%%R

prop.list <- getTransformedProps(colData(sce)$clusters, sample = colData(sce)$sample, transform = "asin")

In [ ]:
%%R

df = data.frame(Proportions = as.vector(t(prop.list$Proportions)), Samples = rep(colnames(prop.list$Proportions), nrow(prop.list$Proportions)), Clusters = rep(rownames(prop.list$Proportions), each=ncol(prop.list$Proportions)))

In [ ]:
%%R -w 4 -h 3 -r 300 --units in

color_dict <- c('OPC-Control'= 'antiquewhite',
 'OPC-A'= '#9e0142',
 'OPC-B'= '#d8434e',
 'OPC-C'= '#f67a49',
 'OPC-D'= '#fdbf6f'#,
 #'OLG-Control'= 'lightgray',
 #'OLG-B'= '#bfe5a0',
 #'OLG-C'= '#74c7a5',
 #'OLG-D'= '#378ebb',
 #'COPs'= '#f1f9a9'
)

df$Clusters <- factor(df$Clusters, levels = rev(c('OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D'))) #, 'OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D', 'COPs')))

ggplot(df, aes(x = Samples, y = Proportions, fill = Clusters)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = color_dict, guide = NULL) +
  scale_x_discrete(limits = c('Control A', 'Control B', 'Control C', 'CAR T A', 'CAR T B', 'CAR T C', 'CAR T D')) + 
  scale_y_continuous(expand = c(0, 0)) +  # Optional: remove space between bars and x-axis
  theme_minimal() + theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
# calculate cell proportions per sample
cell_props = pd.crosstab(opc.obs['final_clusters'], opc.obs['sampleid']).apply(lambda col: col / sum(col))
cell_props

In [ ]:
%%R -i cell_props
# import cell proportions dataframe into R and format for making plots
cell_props$celltype <- rownames(cell_props)

cell_props_df <- reshape2::melt(cell_props)

colnames(cell_props_df) <- c("celltype", "sample", "proportion")

cell_props_df <- cell_props_df %>% mutate("condition" = ifelse(sample %in% c("CART_A", "CART_B", "CART_C", "CART_D"), "CAR T", "Control"))

cell_props_df$condition <- factor(cell_props_df$condition, levels = c("Control", "CAR T"))


In [ ]:
%%R
# calculate mean of each group
stats_df = cell_props_df %>% filter(celltype == 'OPC-Control') %>% mutate("percent" = proportion*100) %>% group_by(condition) %>% summarise(percent = mean(percent))

stats_df

In [ ]:
%%R -w 6 -h 2 -r 300 --units in
# 
# plot OPC-Control cell proportions between conditions
set.seed(111)
p1 <- ggplot(cell_props_df %>% filter(celltype == 'OPC-Control') %>% mutate("percent" = proportion*100), 
        aes(x=condition, y=percent, fill=condition)) +
          geom_jitter(size = 3, width = 0.1, pch=21) + 
            geom_crossbar(data=stats_df, aes(ymin = percent, ymax = percent, color = condition),
                  size=0.5, width = 0.5) + 
            labs(y = "Percentage of OPCs per donor in OPC-Control", x = "") +
            scale_y_continuous(labels = function(x) paste0(x, "%"), limits = c(0, 105)) + 
                scale_fill_manual(values = c("#9aceeb", "#dc143c")) +
                scale_color_manual(values = c("#9aceeb", "#dc143c")) +
                theme_pubr() + theme(axis.text.x = element_text(size = 14, angle = 0), axis.text.y = element_text(size = 12),
                axis.title.y = element_text(size = 12)) + guides(color = "none", fill = "none") + coord_flip()

p1 

In [ ]:
olg = oligos[oligos.obs['final_clusters'].isin(['OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D'])]

In [ ]:
olg

In [ ]:
# extract metadata for creating SCE object
cluster_labels = olg.obs['final_clusters']
sample_labels = olg.obs['sampleid_nounderscore']
group_labels = olg.obs['condition']

In [ ]:
%%R -i cluster_labels -i sample_labels -i group_labels
# create SCE object
sce <- SingleCellExperiment(list(counts=matrix(ncol = length(sample_labels), nrow = 1)),
                     colData=data.frame(clusters=cluster_labels,
                                        sample=sample_labels,
                                        group=group_labels))

In [ ]:
%%R
# Run propeller testing for cell type proportion differences between the two 
# groups
propeller_results <- propeller(clusters = sce$clusters, sample = sce$sample, 
group = sce$group, transform = "asin")

propeller_results

In [ ]:
%%R

saveRDS(propeller_results, "../output/human/human_oligos_OLGs_propeller_differentialabundance_test_results.rds")
write.csv(propeller_results, "../output/human/human_oligos_OLGs_propeller_differentialabundance_test_results.csv")

In [ ]:
%%R

prop.list <- getTransformedProps(colData(sce)$clusters, sample = colData(sce)$sample, transform = "asin")

In [ ]:
%%R

df = data.frame(Proportions = as.vector(t(prop.list$Proportions)), Samples = rep(colnames(prop.list$Proportions), nrow(prop.list$Proportions)), Clusters = rep(rownames(prop.list$Proportions), each=ncol(prop.list$Proportions)))

In [ ]:
%%R -w 4 -h 3 -r 300 --units in

color_dict <- c(#'OPC-Control'= 'antiquewhite',
 #'OPC-A'= '#9e0142',
 #'OPC-B'= '#d8434e',
 #'OPC-C'= '#f67a49',
 #'OPC-D'= '#fdbf6f'#,
 'OLG-Control'= 'lightgray',
 'OLG-B'= '#bfe5a0',
 'OLG-C'= '#74c7a5',
 'OLG-D'= '#378ebb'#,
 #'COPs'= '#f1f9a9'
)

df$Clusters <- factor(df$Clusters, levels = rev(c(#'OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D',
                    'OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D')))

ggplot(df, aes(x = Samples, y = Proportions, fill = Clusters)) +
  geom_bar(stat = "identity") +
  scale_fill_manual(values = color_dict, guide = NULL) +
  scale_x_discrete(limits = c('Control A', 'Control B', 'Control C', 'CAR T A', 'CAR T B', 'CAR T C', 'CAR T D')) + 
  scale_y_continuous(expand = c(0, 0)) +  # Optional: remove space between bars and x-axis
  theme_minimal() + theme_pubr() + theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1)) + labs(y = "Proportion per Donor", x = "")

In [ ]:
# calculate cell proportions per sample
cell_props = pd.crosstab(olg.obs['final_clusters'], olg.obs['sampleid']).apply(lambda col: col / sum(col))
cell_props

In [ ]:
%%R -i cell_props
# import cell proportions dataframe into R and format for making plots
cell_props$celltype <- rownames(cell_props)

cell_props_df <- reshape2::melt(cell_props)

colnames(cell_props_df) <- c("celltype", "sample", "proportion")

cell_props_df <- cell_props_df %>% mutate("condition" = ifelse(sample %in% c("CART_A", "CART_B", "CART_C", "CART_D"), "CAR T", "Control"))

cell_props_df$condition <- factor(cell_props_df$condition, levels = c("Control", "CAR T"))


In [ ]:
%%R
# calculate mean of each group
stats_df = cell_props_df %>% filter(celltype == 'OLG-Control') %>% mutate("percent" = proportion*100) %>% group_by(condition) %>% summarise(percent = mean(percent))

stats_df

In [ ]:
%%R -w 6 -h 2 -r 300 --units in
# 
# plot OLG-Control cell proportions between conditions
set.seed(111)
p1 <- ggplot(cell_props_df %>% filter(celltype == 'OLG-Control') %>% mutate("percent" = proportion*100), 
        aes(x=condition, y=percent, fill=condition)) +
          geom_jitter(size = 3, width = 0.1, pch=21) + 
            geom_crossbar(data=stats_df, aes(ymin = percent, ymax = percent, color = condition),
                  size=0.5, width = 0.5) + 
            labs(y = "Percentage of mature oligodendrocytes per donor in OLG-Control", x = "") +
            scale_y_continuous(labels = function(x) paste0(x, "%"), limits = c(0, 105)) + 
                scale_fill_manual(values = c("#9aceeb", "#dc143c")) +
                scale_color_manual(values = c("#9aceeb", "#dc143c")) +
                theme_pubr() + theme(axis.text.x = element_text(size = 14, angle = 0), axis.text.y = element_text(size = 12),
                axis.title.y = element_text(size = 12)) + guides(color = "none", fill = "none") + coord_flip()

p1 

In [ ]:
# load scanpro sample-level DA test results
OPC_scanpro_df = pd.read_pickle("../output/human/human_oligos_scanpro_differentialabundance_test_results_OPCs.pkl")
OLG_scanpro_df = pd.read_pickle("../output/human/human_oligos_scanpro_differentialabundance_test_results_OLGs.pkl")

In [ ]:
scanpro_df = pd.concat([OPC_scanpro_df, OLG_scanpro_df])

In [ ]:
%%R -i scanpro_df
# reshape to matrix
scanpro_mat = reshape2::acast(scanpro_df, Sample ~ clusters, value.var = "adjusted_p_values")

In [ ]:
%%R
# fill NA values
scanpro_mat[is.na(scanpro_mat)] <- 1

In [ ]:
%%R
# create significance matrices for heatmap annotation
up_sig_mat = scanpro_df %>% mutate(adjusted_p_values = ifelse((prop_ratio > 1) & (adjusted_p_values < 0.01), adjusted_p_values, 1))
up_sig_mat = reshape2::acast(up_sig_mat, Sample~clusters, value.var = "adjusted_p_values")
up_sig_mat = t(up_sig_mat)
up_sig_mat[is.na(up_sig_mat)] <- 1
down_sig_mat = scanpro_df %>% mutate(adjusted_p_values = ifelse((prop_ratio < 1) & (adjusted_p_values < 0.01), adjusted_p_values, 1))
down_sig_mat = reshape2::acast(down_sig_mat, Sample~clusters, value.var = "adjusted_p_values")
down_sig_mat = t(down_sig_mat)
down_sig_mat[is.na(down_sig_mat)] <- 1
up_sig_mat = t(up_sig_mat)
down_sig_mat = t(down_sig_mat)

In [ ]:
%%R -w 4.25 -h 3.5 -r 300 --units in
# plot heatmap

hmap = Heatmap(name = '-log10(p-value)', t(-log10(t(scanpro_mat))), col = circlize::colorRamp2(breaks = c(0, 1, 2.5, 5), 
                                colors=RColorBrewer::brewer.pal(4, "Purples")),
                            cluster_rows = F, cluster_columns = T, 
                            clustering_distance_columns = "pearson",
                        cell_fun = function(j, i, x, y, w, h, fill){
                                if(up_sig_mat[i, j] < 0.01){
                                	gb = textGrob("↑")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                	grid.text("↑", x, y - gb_h*0.15 + gb_w*0.4, gp = gpar(fontsize = 18, fontface = "bold"))
                                } else if(down_sig_mat[i, j] < 0.01){
                                	gb = textGrob("↓")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                	grid.text("↓", x, y - gb_h*0.15 + gb_w*0.4, gp = gpar(fontsize = 18, fontface = "bold"))
                                } else {
                                    gb = textGrob("")
                                	gb_w = convertWidth(grobWidth(gb), "mm")
                                	gb_h = convertHeight(grobHeight(gb), "mm")
                                    grid.text("", x, y - gb_h*0.5 + gb_w*0.4)
                                }
                            }, 
                        heatmap_legend_param = list(direction = "horizontal", position = "left",
                                                           title_position = "topcenter"
                                                           )
                        )

draw(hmap, heatmap_legend_side = "top")

In [ ]:
# create color map for gene expression plots
cmap = mpl.cm.gist_heat_r(np.linspace(0,1,1000))
cmap = mpl.colors.ListedColormap(cmap[25:,:-1])

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "PDGFRA"

with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap, colorbar_loc=None,
           vmin = 0,frameon=False, title='', legend_loc = None)

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "PDGFRA"

with rc_context({"figure.figsize": (4, 2), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap,
           vmin = 0,frameon=False, title='')

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "BCAS1"

with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap, colorbar_loc=None,
           vmin = 0,frameon=False, title='', legend_loc = None)

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "BCAS1"

with rc_context({"figure.figsize": (4, 2), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap,
           vmin = 0,frameon=False, title='')

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "MOBP"

with rc_context({"figure.figsize": (3, 3), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap, colorbar_loc=None,
           vmin = 0,frameon=False, title='', legend_loc = None)

In [ ]:
# plot UMAP with OPC marker genes overlaid

GENE = "MOBP"

with rc_context({"figure.figsize": (4, 2), "grid.alpha":0}):
    sc.pl.umap(oligos, color=[GENE], size = 25,
          cmap = cmap,
           vmin = 0,frameon=False, title='')

In [ ]:
oligos.write('../output/human/human_oligodendrocytes_final_annotated_scanpy_object.h5ad')

In [ ]:
# now, run pseudobulk DE testing comparing our CAR T clusters to OPC/OLG-Control clusters

In [ ]:
oligos.X = oligos.layers['raw_counts']

In [ ]:
# examine the cell counts in each cluster from each sample
pd.crosstab(oligos.obs['final_clusters'], oligos.obs['sampleid'])

In [ ]:
# examine the proportion of cells in each cluster from each sample
pd.crosstab(oligos.obs['final_clusters'], oligos.obs['sampleid']).apply(lambda col: col / sum(col))

In [ ]:
# define function for creating sample pseudobulk aggregates
# adapted from https://www.sc-best-practices.org/conditions/differential_gene_expression.html#pseudobulk
# in this implementation, clusters of large enough size will be automatically split into two or three clusters
# if the number of samples in a the cluster are not ≥3. Note that this is not strictly 'pseudo-bulk'
# DE testing, and will include 'pseudoreplication' in the event that a cluster must be split. 
# We drew this idea from work showing even single sample scRNA-seq datasets benefit from creating pseudo-bulks
# for DE testing (https://doi.org/10.1101/2023.03.28.534443)

import random
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="sampleid",
    cell_identity_key="cell_type",
    obs_to_keep=['sampleid', 'condition', 'batch', 'sex', 'age'], 
    replicates_per_patient=1,
):
    
    print("Pseudobulk aggregating group " + cell_identity + " ...")

    random.seed(10)
    
    # subset adata to the given cell identity
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby(by=donor_key).size()
    donors_to_drop = []
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]

    donors_to_keep = [x for x in size_by_donor.index if x not in donors_to_drop]

    n_donors = len([i for i in donors_to_keep])

    prop_df = pd.crosstab(adata.obs[cell_identity_key], adata.obs[donor_key]).apply(lambda col: col / sum(col))
    sig_samples = prop_df.columns[(prop_df[prop_df.index == cell_identity] > 0.1).all()].values

    if n_donors > 1:
        donors_to_keep2 = [x for x in donors_to_keep if x in sig_samples]
    else:
        donors_to_keep2 = donors_to_keep
    
    print("Keeping donors:")
    print(donors_to_keep2)
    
    n_samples = len([i for i in donors_to_keep2])

    split2x = []
    split1x = []

    if n_samples >= 3:
        print("Cluster has 3+ samples. Proceeding.")
    
    if n_samples == 2:
        print("Less than 3 samples in cluster. Attempting to subdivide largest sample into 2 sub-samples...")
        
        largest_sample = size_by_donor[[i for i in size_by_donor.index]].idxmax()
        
        if size_by_donor[largest_sample] >= (2*NUM_OF_CELL_PER_DONOR):
            print("Splitting sample: ")
            print(largest_sample)
            split1x.append(largest_sample)

        else:
            print("No sample in this cluster is large enough to split. Proceeding with 2 samples only.")
            print("WARNING: DE TESTING WILL BE UNDER-POWERED.")
            
    if n_samples == 1:
        print("Only 1 sample is in this cluster. Attempting to subdivide this sample into 3 sub-samples...")
    
        solo_sample = [i for i in donors_to_keep][0]
        
        if size_by_donor[solo_sample] >= (3*NUM_OF_CELL_PER_DONOR):
            print("Splitting sample into 3 sub-samples: ")
            print(solo_sample)
            split2x.append(solo_sample)
        elif size_by_donor[solo_sample] >= (2*NUM_OF_CELL_PER_DONOR):
            print("Sample is not large enough to split into 3 sub-samples. Splitting sample into 2 sub-samples:")
            print(solo_sample)
            print("WARNING: DE TESTING WILL BE UNDER-POWERED.")
            split1x.append(solo_sample)
        else:
            print("Sample is too small to by sub-divided:")
            print(solo_sample)
            print("WARNING: CANNOT USE THIS SINGLE PSEUDOBULK FOR DE TESTING.")

    if n_samples == 0:
        print("This cluster has NO samples of appropriate size...")
    
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop = adata_cell_pop[adata_cell_pop.obs[donor_key].isin(donors_to_keep2)]

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor in donors_to_keep2:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            if donor in split2x:
                replicates_per_patient = 3
            elif donor in split1x:
                replicates_per_patient = 2
            else:
                replicates_per_patient = 1
                
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                df_donor = pd.DataFrame(adata_replicate.layers['raw_counts'].A)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(by=donor_key).agg(agg_dict)
                df_donor[donor_key] = donor
                df.loc[f"donor_{donor}_{i}"] = df_donor.loc[donor]
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)   
    )

    adata_cell_pop.obs['cluster'] = cell_identity 
    
    return adata_cell_pop

In [ ]:
# run pseudobulk aggregation
NUM_OF_CELL_PER_DONOR = 50

OPCa_pbs = aggregate_and_filter(oligos, cell_identity = 'OPC-A', cell_identity_key = "final_clusters")
OPCb_pbs = aggregate_and_filter(oligos, cell_identity = 'OPC-B', cell_identity_key = "final_clusters")
OPCc_pbs = aggregate_and_filter(oligos, cell_identity = 'OPC-C', cell_identity_key = "final_clusters")
OPCd_pbs = aggregate_and_filter(oligos, cell_identity = 'OPC-D', cell_identity_key = "final_clusters")
OPCcontrol_pbs = aggregate_and_filter(oligos, cell_identity = 'OPC-Control', cell_identity_key = "final_clusters")

OLGb_pbs = aggregate_and_filter(oligos, cell_identity = 'OLG-B', cell_identity_key = "final_clusters")
OLGc_pbs = aggregate_and_filter(oligos, cell_identity = 'OLG-C', cell_identity_key = "final_clusters")
OLGd_pbs = aggregate_and_filter(oligos, cell_identity = 'OLG-D', cell_identity_key = "final_clusters")
OLGcontrol_pbs = aggregate_and_filter(oligos, cell_identity = 'OLG-Control', cell_identity_key = "final_clusters")

In [ ]:
## concatenate all pseudobulk anndatas
oligos_pbs = ad.concat([OPCa_pbs,
                       OPCb_pbs,
                       OPCc_pbs,
                       OPCd_pbs,
                       OPCcontrol_pbs,
                       
                       OLGb_pbs,
                       OLGc_pbs,
                       OLGd_pbs,
                       OLGcontrol_pbs])
oligos_pbs.obs_names_make_unique()

In [ ]:
oligos_pbs

In [ ]:
oligos_pbs.obs

In [ ]:
# extract count matrix and metadata info for passing to R
input_matrix = oligos_pbs.X
gene_ids = oligos_pbs.var_names.values
condition_ids = oligos_pbs.obs['condition']
cluster_ids = oligos_pbs.obs['cluster']
sample_ids = oligos_pbs.obs['sampleid'].values
batch_ids = oligos_pbs.obs['batch'].values
sex_ids = oligos_pbs.obs['sex'].values

In [ ]:
%%R -i input_matrix -i gene_ids -i sample_ids -i batch_ids -i cluster_ids -i condition_ids -i sex_ids
# import to R and check dimensions
colnames(input_matrix) <- paste(sample_ids, cluster_ids, sep = "_")
rownames(input_matrix) <- gene_ids
# create SCE object
oligos_pbs <- SingleCellExperiment(list('counts'=input_matrix))
# add metadata to SCE object
colData(oligos_pbs)$sample <- sample_ids
colData(oligos_pbs)$cluster <- cluster_ids
colData(oligos_pbs)$batch <- batch_ids
colData(oligos_pbs)$condition <- condition_ids
colData(oligos_pbs)$sex <- sex_ids

In [ ]:
%%R

oligos_pbs

In [ ]:
%%R

opc_pbs = oligos_pbs[,colData(oligos_pbs)$cluster %in% c('OPC-A', 'OPC-B', 'OPC-C', 'OPC-D', 'OPC-Control')]

In [ ]:
%%R

colData(opc_pbs)

In [ ]:
%%R
# set factor levels for cluster metadata
colData(opc_pbs)$cluster <- factor(colData(opc_pbs)$cluster, levels = c('OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D'))

In [ ]:
%%R
# define function for edgeR differential expression testing
fit_model <- function(adata_){

    set.seed(1)
    
    # create an edgeR object with counts and grouping factor
    group <- colData(adata_)$cluster
    batch <- colData(adata_)$batch
    sex <- colData(adata_)$sex

    y <- DGEList(assay(adata_, "counts"), group = group)

    # filter out genes with low counts
    print("Dimensions before subsetting:")
    print(dim(y))
    print("")
    keep <- filterByExpr(y)
    y <- y[keep, , keep.lib.sizes=FALSE]
    print("Dimensions after subsetting:")
    print(dim(y))
    print("")

    # normalize
    y <- calcNormFactors(y)
    # create a design matrix:
    design <- model.matrix(~ group + batch + sex)
    # estimate dispersion
    y <- estimateDisp(y, design = design)
    # fit the model
    fit <- glmQLFit(y, design)
    return(list("fit"=fit, "design"=design, "y"=y))
}

In [ ]:
%%time
%%R
# run edgeR
outs <-fit_model(opc_pbs)

In [ ]:
%%R
# extract results
fit <- outs$fit
y <- outs$y

In [ ]:
%%R -w 12 -h 5 --units in -r 300
# create MDS plot
plotMDS(y, col=ifelse(y$samples$group == "OPC-Control", "blue", "red"))

In [ ]:
%%R

plotQLDisp(fit)

In [ ]:
%%R

# save output
saveRDS(opc_pbs, "../output/human/human_opc_edgeR_pseudobulk_sce_object.rds")

In [ ]:
%%R
# save output
saveRDS(outs, "../output/human/human_opc_edgeR_glmqlf_output_object.rds")

In [ ]:
%%R -o edgeR_filtered_genes

edgeR_filtered_genes = rownames(outs$y$counts)

In [ ]:
np.save('../output/human/human_opc_edgeR_filtered_genes.npy', edgeR_filtered_genes)

In [ ]:
# Test OPC-A versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OPCa <- glmQLFTest(fit, coef="groupOPC-A")

In [ ]:
%%R -o tt_OPCa
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OPCa <- topTags(qlf_OPCa, n = Inf)
tt_OPCa <- tt_OPCa$table

In [ ]:
# view DE results
tt_OPCa[(tt_OPCa.FDR < 0.05/7) & (abs(tt_OPCa.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OPCa, "../output/human/human_opc_edgeR_glmqlftest_output_object_opca-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OPCa.to_csv("../output/human/human_opc-A_v_opc-control_edgeR_results.csv")
tt_OPCa.to_pickle("../output/human/human_opc-A_v_opc-control_edgeR_results.pkl")
tt_OPCa[(tt_OPCa.FDR < 0.05/7) & (abs(tt_OPCa.logFC) > 1)].to_csv("../output/human/human_opc-A_v_opc-control_edgeR_significant_results.csv")
tt_OPCa[(tt_OPCa.FDR < 0.05/7) & (abs(tt_OPCa.logFC) > 1)].to_pickle("../output/human/human_opc-A_v_opc-control_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OPCa, de.tags = rownames(tt_OPCa)[which((tt_OPCa$FDR<0.05/7) & (abs(tt_OPCa$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('CR1', 'MYL6B', 'MYL12B', 'CNR1', 'PLAAT3', 'HLA-DMA', 'B2M', "GPNMB", "FOS", "C1QTNF7",
                  "GRIA4", "SEMA3E", "MARCKS", "ABLIM1", "STK32B", "CHRDL1", "CDH22"
                  
                 )


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OPCa %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .75, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test OPC-B versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OPCb <- glmQLFTest(fit, coef="groupOPC-B")

In [ ]:
%%R -o tt_OPCb
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OPCb <- topTags(qlf_OPCb, n = Inf)
tt_OPCb <- tt_OPCb$table

In [ ]:
# view DE results
tt_OPCb[(tt_OPCb.FDR < 0.05/7) & (abs(tt_OPCb.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OPCb, "../output/human/human_opc_edgeR_glmqlftest_output_object_OPCb-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OPCb.to_csv("../output/human/human_opc-B_v_opc-control_edgeR_results.csv")
tt_OPCb.to_pickle("../output/human/human_opc-B_v_opc-control_edgeR_results.pkl")
tt_OPCb[(tt_OPCb.FDR < 0.05/7) & (abs(tt_OPCb.logFC) > 1)].to_csv("../output/human/human_opc-B_v_opc-control_edgeR_significant_results.csv")
tt_OPCb[(tt_OPCb.FDR < 0.05/7) & (abs(tt_OPCb.logFC) > 1)].to_pickle("../output/human/human_opc-B_v_opc-control_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OPCb, de.tags = rownames(tt_OPCb)[which((tt_OPCb$FDR<0.05/7) & (abs(tt_OPCb$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('ATF4', 'DDIT3', 'CEBPZ', 'CEBPG', "BAG", "HSPD1", "DNAJB1", "HSPA6", 'CHORDC1', 'XPO1', 'HMGB1', 'CIITA', 'HLA-A',
                 'OPCML', 'KCNIP1', 'TNR', 'TRPM3', 'CNTN5', 'UNC5D', "CHST8", "CPQ")


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OPCb %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test OPC-C versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OPCc <- glmQLFTest(fit, coef="groupOPC-C")

In [ ]:
%%R -o tt_OPCc
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OPCc <- topTags(qlf_OPCc, n = Inf)
tt_OPCc <- tt_OPCc$table

In [ ]:
# view DE results
tt_OPCc[(tt_OPCc.FDR < 0.05/7) & (abs(tt_OPCc.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OPCc, "../output/human/human_opc_edgeR_glmqlftest_output_object_OPCc-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OPCc.to_csv("../output/human/human_opc-C_v_opc-control_edgeR_results.csv")
tt_OPCc.to_pickle("../output/human/human_opc-C_v_opc-control_edgeR_results.pkl")
tt_OPCc[(tt_OPCc.FDR < 0.05/7) & (abs(tt_OPCc.logFC) > 1)].to_csv("../output/human/human_opc-C_v_opc-control_edgeR_significant_results.csv")
tt_OPCc[(tt_OPCc.FDR < 0.05/7) & (abs(tt_OPCc.logFC) > 1)].to_pickle("../output/human/human_opc-C_v_opc-control_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OPCc, de.tags = rownames(tt_OPCc)[which((tt_OPCc$FDR<0.05/7) & (abs(tt_OPCc$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('PRANCR', 'GPC5', 'IGF2BP2', 'APIP', 'NAIP', 'NMT1', 'DDX3X', 'BCL2L11', 'MERTK',
                 'ITGA8', 'GALNTL6', 'DOCK5', "ABLIM1", "AJAP1", "CAMK4")


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OPCc %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test OPC-D versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OPCd <- glmQLFTest(fit, coef="groupOPC-D")

In [ ]:
%%R -o tt_OPCd
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OPCd <- topTags(qlf_OPCd, n = Inf)
tt_OPCd <- tt_OPCd$table

In [ ]:
# view DE results
tt_OPCd[(tt_OPCd.FDR < 0.05/7) & (abs(tt_OPCd.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OPCd, "../output/human/human_opc_edgeR_glmqlftest_output_object_OPCd-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OPCd.to_csv("../output/human/human_opc-D_v_opc-control_edgeR_results.csv")
tt_OPCd.to_pickle("../output/human/human_opc-D_v_opc-control_edgeR_results.pkl")
tt_OPCd[(tt_OPCd.FDR < 0.05/7) & (abs(tt_OPCd.logFC) > 1)].to_csv("../output/human/human_opc-D_v_opc-control_edgeR_significant_results.csv")
tt_OPCd[(tt_OPCd.FDR < 0.05/7) & (abs(tt_OPCd.logFC) > 1)].to_pickle("../output/human/human_opc-D_v_opc-control_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OPCd, de.tags = rownames(tt_OPCd)[which((tt_OPCd$FDR<0.05/7) & (abs(tt_OPCd$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('MSMO1', 'FDFT1', 'IL1RAP', 'HMGCS1', 'HMGCR', 'SREBF2', 'HLA-B', 'HLA-F', 'B2M',
                 'EYA4', 'MYRFL', 'ITGA8', "CDH10", 'PLCG2', 'SGCZ', 'ABLIM1', 'DOCK5', 'VSTM2A')


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OPCd %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
%%R


OPC_edgeR_df <- do.call("rbind", list(
    as.data.frame(tt_OPCa) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-A"),
    as.data.frame(tt_OPCb) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-B"),
    as.data.frame(tt_OPCc) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-C"),
    as.data.frame(tt_OPCd) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-D")
    ))


OPC_sig_degs_df <- do.call("rbind", list(
    as.data.frame(tt_OPCa) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-A") %>% filter(FDR < 0.05/7),
    as.data.frame(tt_OPCb) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-B") %>% filter(FDR < 0.05/7),
    as.data.frame(tt_OPCc) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-C") %>% filter(FDR < 0.05/7),
    as.data.frame(tt_OPCd) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OPC-D") %>% filter(FDR < 0.05/7)
    ))

OPC_up_degs <- list(
    "OPC-A"=as.data.frame(tt_OPCa) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-B"=as.data.frame(tt_OPCb) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-C"=as.data.frame(tt_OPCc) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-D"=as.data.frame(tt_OPCd) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.)
    )

OPC_down_degs <- list(
    "OPC-A"=as.data.frame(tt_OPCa) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-B"=as.data.frame(tt_OPCb) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-C"=as.data.frame(tt_OPCc) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OPC-D"=as.data.frame(tt_OPCd) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.)
    )


In [ ]:
%%R -o ann_sig_df
# examine the number of DEGs in each cluster compared to OPC-control
ann_sig_df = OPC_sig_degs_df %>% mutate(annotation = ifelse(logFC > 0, "Up-regulated", "Down-regulated"))

In [ ]:
data = pd.crosstab(ann_sig_df.cluster, ann_sig_df.annotation)

In [ ]:
data

In [ ]:
font_color = '#525252'
hfont = {'fontname':'Arial'}
facecolor = '#eaeaf2'
color1 = '#29ABE2'
color2 = '#ED1C24'
index = data.index
column0 = data['Down-regulated']
column1 = data['Up-regulated']
title0 = 'Downregulated'
title1 = 'Upregulated'

fig, axes = plt.subplots(figsize=(10,8), ncols=2, sharey=True)
fig.tight_layout()

axes[0].barh(index, column0, align='center', color=color1, zorder=10)
axes[0].set_title(title0, fontsize=25, pad=15, color=color1, fontweight="bold", **hfont)
axes[1].barh(index, column1, align='center', color=color2, zorder=10)
axes[1].set_title(title1, fontsize=25, pad=15, color=color2, fontweight="bold", **hfont)

# If you have positive numbers and want to invert the x-axis of the left plot
axes[0].invert_xaxis() 

# To show data from highest to lowest
plt.gca().invert_yaxis()

axes[0].set(yticks=data.index, yticklabels=data.index)
axes[0].yaxis.tick_left()
axes[0].tick_params(axis='y', colors='white') # tick color

# Hide tick marks
axes[1].tick_params(axis='y', bottom=False, top=False, left=False, right=False)

# Show only tick labels
plt.xticks(visible=True)
plt.yticks(visible=True)



for label in (axes[0].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[0].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)
for label in (axes[1].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[1].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)

axes[0].grid(False)
axes[1].grid(False)

plt.subplots_adjust(wspace=0, top=0.85, bottom=0.1, left=0.18, right=0.95)


In [ ]:
%%R

OPC_up_mat = make_comb_mat(OPC_up_degs)
OPC_down_mat = make_comb_mat(OPC_down_degs)

In [ ]:
%%R

head(OPC_up_mat)

In [ ]:
%%R

head(OPC_down_mat)

In [ ]:
%%R

UpSet(OPC_up_mat)

In [ ]:
%%R

UpSet(OPC_down_mat)

In [ ]:
%%R

print("Up-regulated DEGs common to all samples:")
print(extract_comb(OPC_up_mat, "1111"))
print(noquote(""))
print("Down-regulated DEGs common to all samples:")
print(extract_comb(OPC_down_mat, "1111"))

In [ ]:
from matplotlib.ticker import MaxNLocator

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(oligos, keys = ['FOS'], layer = "log1p_norm", 
                 groupby = "final_clusters",
                 order = ['OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D'],
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'FOS', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(oligos, keys = ['ABLIM1'], layer = "log1p_norm", 
                 groupby = "final_clusters",
                 order = ['OPC-Control', 'OPC-A', 'OPC-B', 'OPC-C', 'OPC-D'],
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'ABLIM1', fontdict={'fontstyle': 'italic'})

In [ ]:
%%R

olg_pbs = oligos_pbs[,colData(oligos_pbs)$cluster %in% c('OLG-B', 'OLG-C', 'OLG-D', 'OLG-Control')]

In [ ]:
%%R

colData(olg_pbs)

In [ ]:
%%R
# set factor levels for cluster metadata
colData(olg_pbs)$cluster <- factor(colData(olg_pbs)$cluster, levels = c('OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D'))

In [ ]:
%%time
%%R
# run edgeR
outs2 <-fit_model(olg_pbs)

In [ ]:
%%R
# extract results
fit2 <- outs2$fit
y2 <- outs2$y

In [ ]:
%%R -w 12 -h 5 --units in -r 300
# create MDS plot
plotMDS(y2, col=ifelse(y2$samples$group == "OLG-Control", "blue", "red"))

In [ ]:
%%R

plotQLDisp(fit2)

In [ ]:
%%R

# save output
saveRDS(olg_pbs, "../output/human/human_olg_edgeR_pseudobulk_sce_object.rds")

In [ ]:
%%R
# save output
saveRDS(outs2, "../output/human/human_olg_edgeR_glmqlf_output_object.rds")

In [ ]:
%%R -o edgeR_filtered_genes

edgeR_filtered_genes = rownames(outs2$y$counts)

In [ ]:
np.save('../output/human/human_olg_edgeR_filtered_genes.npy', edgeR_filtered_genes)

In [ ]:
# Test OLG-B versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OLGb <- glmQLFTest(fit2, coef="groupOLG-B")

In [ ]:
%%R -o tt_OLGb
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OLGb <- topTags(qlf_OLGb, n = Inf)
tt_OLGb <- tt_OLGb$table

In [ ]:
# view DE results
tt_OLGb[(tt_OLGb.FDR < 0.05/7) & (abs(tt_OLGb.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OLGb, "../output/human/human_opc_edgeR_glmqlftest_output_object_OLGb-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OLGb.to_csv("../output/human/human_OLGb_v_OLGcontrol_edgeR_results.csv")
tt_OLGb.to_pickle("../output/human/human_OLGb_v_OLGcontrol_edgeR_results.pkl")
tt_OLGb[(tt_OLGb.FDR < 0.05/7) & (abs(tt_OLGb.logFC) > 1)].to_csv("../output/human/human_OLGb_v_OLGcontrol_edgeR_significant_results.csv")
tt_OLGb[(tt_OLGb.FDR < 0.05/7) & (abs(tt_OLGb.logFC) > 1)].to_pickle("../output/human/human_OLGb_v_OLGcontrol_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OLGb, de.tags = rownames(tt_OLGb)[which((tt_OLGb$FDR<0.05/7) & (abs(tt_OLGb$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('SOD1', 'BAG3', 'DNAJA4', 'SORBS1', 'XPO1', 'HSPA9', 'HSPA4L', 'HIBCH', 'FMN1',
                 'STMN4', 'UNC5C', 'SLC5A11', 'DPP10', 'MOBP', 'MBP', 'MOG', 'DLG2', 'KAZN')


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OLGb %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
            mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test OLG-C versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OLGc <- glmQLFTest(fit2, coef="groupOLG-C")

In [ ]:
%%R -o tt_OLGc
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OLGc <- topTags(qlf_OLGc, n = Inf)
tt_OLGc <- tt_OLGc$table

In [ ]:
# view DE results
tt_OLGc[(tt_OLGc.FDR < 0.05/7) & (abs(tt_OLGc.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OLGc, "../output/human/human_opc_edgeR_glmqlftest_output_object_OLGc-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OLGc.to_csv("../output/human/human_OLGc_v_OLGcontrol_edgeR_results.csv")
tt_OLGc.to_pickle("../output/human/human_OLGc_v_OLGcontrol_edgeR_results.pkl")
tt_OLGc[(tt_OLGc.FDR < 0.05/7) & (abs(tt_OLGc.logFC) > 1)].to_csv("../output/human/human_OLGc_v_OLGcontrol_edgeR_significant_results.csv")
tt_OLGc[(tt_OLGc.FDR < 0.05/7) & (abs(tt_OLGc.logFC) > 1)].to_pickle("../output/human/human_OLGc_v_OLGcontrol_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OLGc, de.tags = rownames(tt_OLGc)[which((tt_OLGc$FDR<0.05/7) & (abs(tt_OLGc$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('CHRM5', 'TMPRSS5', 'PEBP1', 'ENO4', "PTCSC3", 'ADAMTS18', 'SBNO1', 'CNBP',
                 "ADGRL3", "CADM2", 'DLG2', 'CALN1', "ABHD2", 'NLGN3')


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OLGc %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
# Test OLG-D versus OPC-Control

In [ ]:
%%R
# run glmQLFTest
qlf_OLGd <- glmQLFTest(fit2, coef="groupOLG-D")

In [ ]:
%%R -o tt_OLGd
# get all of the DE genes and calculate Benjamini-Hochberg adjusted FDR
tt_OLGd <- topTags(qlf_OLGd, n = Inf)
tt_OLGd <- tt_OLGd$table

In [ ]:
# view DE results
tt_OLGd[(tt_OLGd.FDR < 0.05/7) & (abs(tt_OLGd.logFC) > 1)]

In [ ]:
%%R

saveRDS(qlf_OLGd, "../output/human/human_opc_edgeR_glmqlftest_output_object_OLGd-v-opccontrol.rds")

In [ ]:
# save DE results to CSV and pickle files
tt_OLGd.to_csv("../output/human/human_OLGd_v_OLGdontrol_edgeR_results.csv")
tt_OLGd.to_pickle("../output/human/human_OLGd_v_OLGdontrol_edgeR_results.pkl")
tt_OLGd[(tt_OLGd.FDR < 0.05/7) & (abs(tt_OLGd.logFC) > 1)].to_csv("../output/human/human_OLGd_v_OLGcontrol_edgeR_significant_results.csv")
tt_OLGd[(tt_OLGd.FDR < 0.05/7) & (abs(tt_OLGd.logFC) > 1)].to_pickle("../output/human/human_OLGd_v_OLGcontrol_edgeR_significant_results.pkl")

In [ ]:
%%R
# create log fold change versus average expression plot
plotSmear(qlf_OLGd, de.tags = rownames(tt_OLGd)[which((tt_OLGd$FDR<0.05/7) & (abs(tt_OLGd$logFC) > 1))])

In [ ]:
%%R -w 4 -h 3 --units in -r 300

select_genes <- c('MSMO1', 'FDFT1', 'LSS', 'GADD45B', 'LDLR', 'FOS', 'SREBF2', 'SREBF1',
                 'PLCG2', 'RELN', 'TPM1', 'MT3', 'MCF2L', "STMN4", "PHGDH",'SRSF12')


# plot volcano plot with automatically generated gene labels
p1 <- ggplot(data = tt_OLGd %>% mutate(gene = rownames(.)) %>% mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/7)), gene, NA)) %>% 
             mutate(label = ifelse((abs(logFC) > 1) & (FDR < (0.05/5)) & (gene %in% select_genes), gene, NA)) %>% 
                                                       mutate(significance = ifelse((abs(logFC) > 1) & (FDR < 0.05/7), "Significant", "NS")), 
              aes(x = logFC, y = -log10(FDR), label = label, color = significance)) +
  geom_vline(xintercept = c(-1, 1), col = "gray", linetype = 'dashed') +
  geom_hline(yintercept = -log10(0.05/7), col = "gray", linetype = 'dashed') +
  geom_point(size = 0.25) + scale_colour_manual(name = 'Significance', values = c("black", "red2")) + 
    ggrepel::geom_text_repel(max.overlaps = 30, min.segment.length = unit(0, 'lines'), nudge_y = .5, nudge_x = 0.5, aes(fontface="italic", color = NULL), force = 3, seed = 72,
                bg.color = "white", bg.r = .05) + 
    labs(x = "Log2-fold change") + 
    theme_pubr() + theme(axis.title.x = element_text(size = 14), axis.title.y = element_text(size = 14), text = element_text(family = "Arial")) + 
    guides(color = "none")

p1

In [ ]:
%%R


OLG_edgeR_df <- do.call("rbind", list(
    as.data.frame(tt_OLGb) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-B"),
    as.data.frame(tt_OLGc) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-C"),
    as.data.frame(tt_OLGd) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-D")
    ))


OLG_sig_degs_df <- do.call("rbind", list(
    as.data.frame(tt_OLGb) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-B") %>% filter(FDR < 0.05/7),
    as.data.frame(tt_OLGc) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-C") %>% filter(FDR < 0.05/7),
    as.data.frame(tt_OLGd) %>% filter(abs(logFC) > 1) %>% mutate(gene = rownames(.)) %>% mutate(cluster = "OLG-D") %>% filter(FDR < 0.05/7)
    ))

OLG_up_degs <- list(
    "OLG-B"=as.data.frame(tt_OLGb) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OLG-C"=as.data.frame(tt_OLGc) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OLG-D"=as.data.frame(tt_OLGd) %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% rownames(.)
    )

OLG_down_degs <- list(
    "OLG-B"=as.data.frame(tt_OLGb) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OLG-C"=as.data.frame(tt_OLGc) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.),
    "OLG-D"=as.data.frame(tt_OLGd) %>% filter(logFC < -1) %>% filter(FDR < 0.05/7) %>% rownames(.)
    )


In [ ]:
%%R -o ann_sig_df
# examine the number of DEGs in each cluster compared to OLG-control
ann_sig_df = OLG_sig_degs_df %>% mutate(annotation = ifelse(logFC > 1, "Up-regulated", "Down-regulated"))

In [ ]:
data = pd.crosstab(ann_sig_df.cluster, ann_sig_df.annotation)

In [ ]:
data

In [ ]:
font_color = '#525252'
hfont = {'fontname':'Arial'}
facecolor = '#eaeaf2'
color1 = '#29ABE2'
color2 = '#ED1C24'
index = data.index
column0 = data['Down-regulated']
column1 = data['Up-regulated']
title0 = 'Downregulated'
title1 = 'Upregulated'

fig, axes = plt.subplots(figsize=(10,8), ncols=2, sharey=True)
fig.tight_layout()

axes[0].barh(index, column0, align='center', color=color1, zorder=10)
axes[0].set_title(title0, fontsize=25, pad=15, color=color1, fontweight="bold", **hfont)
axes[1].barh(index, column1, align='center', color=color2, zorder=10)
axes[1].set_title(title1, fontsize=25, pad=15, color=color2, fontweight="bold", **hfont)

# If you have positive numbers and want to invert the x-axis of the left plot
axes[0].invert_xaxis() 

# To show data from highest to lowest
plt.gca().invert_yaxis()

axes[0].set(yticks=data.index, yticklabels=data.index)
axes[0].yaxis.tick_left()
axes[0].tick_params(axis='y', colors='white') # tick color

# Hide tick marks
axes[1].tick_params(axis='y', bottom=False, top=False, left=False, right=False)

# Show only tick labels
plt.xticks(visible=True)
plt.yticks(visible=True)



for label in (axes[0].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[0].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)
for label in (axes[1].get_xticklabels()):
    label.set(fontsize=15, color='black', **hfont)
for label in (axes[1].get_yticklabels()):
    label.set(fontsize=22, color='black', fontweight="bold", **hfont)

axes[0].grid(False)
axes[1].grid(False)

plt.subplots_adjust(wspace=0, top=0.85, bottom=0.1, left=0.18, right=0.95)


In [ ]:
%%R

OLG_up_mat = make_comb_mat(OLG_up_degs)
OLG_down_mat = make_comb_mat(OLG_down_degs)

In [ ]:
%%R

head(OLG_up_mat)

In [ ]:
%%R

head(OLG_down_mat)

In [ ]:
%%R

UpSet(OLG_up_mat)

In [ ]:
%%R

UpSet(OLG_down_mat)

In [ ]:
%%R

print("Up-regulated DEGs common to all samples:")
print(extract_comb(OLG_up_mat, "111"))
print(noquote(""))
print("Down-regulated DEGs common to all samples:")
print(extract_comb(OLG_down_mat, "111"))

In [ ]:
from matplotlib.ticker import MaxNLocator

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(oligos, keys = ['ADGRL3'], layer = "log1p_norm", 
                 groupby = "final_clusters",
                 order = ['OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D'],
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'ADGRL3', fontdict={'fontstyle': 'italic'})

In [ ]:
with plt.rc_context({"axes.grid": False, "figure.figsize":[6, 1.5]}):
    ax = sc.pl.violin(oligos, keys = ['STMN4'], layer = "log1p_norm", 
                 groupby = "final_clusters",
                 order = ['OLG-Control', 'OLG-B', 'OLG-C', 'OLG-D'],
                 xlabel = None,
                 size = 0.5,use_raw = False, show = False)
    ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins = 2))
    plt.ylabel(ylabel = 'STMN4', fontdict={'fontstyle': 'italic'})

In [ ]:
%%R

up_degs_OPCs <- unique(OPC_edgeR_df %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% group_by(cluster) %>% top_n(50, -log10(FDR)) %>% .$gene)

In [ ]:
%%R

additional_degs <- c("CR1", "PLAAT3", "MYL6B", "MYL12B",
                "ATF4", "DDIT3", "CEBPG", "CEBPZ", 'HSPE1', 'HSPH1',
                "APIP", "NAIP", "NMT1",
                "MSMO1", "SREBF2", "HMGCS1", "HMGCR", "FDFT1",
                "HLA-A", "HLA-B", "HLA-F", "HLA-DMA", "B2M", "CIITA")

In [ ]:
# define function which calculates average expression of a set of genes across cells grouped by a metadata category
def grouped_obs_mean(adata, group_key, layer=None, gene_symbols=None):
    if layer is not None:
        getX = lambda x: x.layers[layer]
    else:
        getX = lambda x: x.X
    if gene_symbols is not None:
        new_idx = adata.var[idx]
    else:
        new_idx = adata.var_names

    grouped = adata.obs.groupby(group_key)
    out = pd.DataFrame(
        np.zeros((adata.shape[1], len(grouped)), dtype=np.float64),
        columns=list(grouped.groups.keys()),
        index=adata.var_names
    )

    for group, idx in grouped.indices.items():
        X = getX(adata[idx])
        out[group] = np.ravel(X.mean(axis=0, dtype=np.float64)).tolist()
    return out

In [ ]:
%%R -o up_degs_OPCs

up_degs_OPCs <- unique(c(up_degs_OPCs, additional_degs))

In [ ]:
# calculate average expression of set of marker genes across choir clusters
OPC_avg_deg_exp = grouped_obs_mean(opc[:, up_degs_OPCs], group_key='final_clusters', layer = 'log1p_norm')

In [ ]:
%%R -i OPC_avg_deg_exp

hmap_mat = t(scale(t(OPC_avg_deg_exp)))

In [ ]:
%%R
# create gene annotation highlighting select genes
selectGenes <- c("CR1", "PLAAT3", "MYL6B", "AP1S2",
                "ATF4", "DDIT3", "CEBPG", "CEBPZ", 'HSPE1', 'HSPH1',
                "APIP", "NAIP", "NMT1",
                "MSMO1", "SREBF2", "HMGCS1", "HMGCR", "FDFT1",
                "HLA-A", "HLA-B", "HLA-F", "HLA-DMA", "B2M", "CIITA")

selectGenes <- selectGenes[order(match(selectGenes,rownames(hmap_mat)))]

ha = rowAnnotation(genes = anno_mark(at = which(rownames(hmap_mat) %in% selectGenes), 
    labels = selectGenes, labels_gp = gpar(col = "black", fontfamily = "Arial", 
                                           fontface="italic", fontsize = 10)))

In [ ]:
%%R -w 5 -h 8 -r 300 --units in
# plot heatmap
set.seed(123)

hmap = Heatmap(name = "Scaled Avg. Exp.", hmap_mat, col = circlize::colorRamp2(breaks = c(-1.5, -0.75, 0, 0.75, 1.5),
                                                                #colors=RColorBrewer::brewer.pal(5, "BuPu")),
                                                                colors=viridis::viridis_pal(option = "magma")(5)),
        right_annotation = ha,
        show_row_names = FALSE,
        cluster_rows = T,
        cluster_columns = T,
        #row_names_gp = grid::gpar(fontsize = 2),
        heatmap_legend_param = list(direction = "horizontal",
                                   title_position = "topcenter"
                                   )
)

draw(hmap)

In [ ]:
%%R

up_degs_OLGs <- unique(OLG_edgeR_df %>% filter(logFC > 1) %>% filter(FDR < 0.05/7) %>% group_by(cluster) %>% top_n(50, -log10(FDR)) %>% .$gene)

In [ ]:
%%R

additional_degs <- c('MBP', 'MOBP', 'MOG',
                                    'HSPA9', 'DNAJA4', "BAG3",
                                    'CHRM5', 'TMPRSS5', 'ADAMTS18', "PEBP1",
                                    'MSMO1', 'FDFT1', 'LSS')

In [ ]:
%%R -o up_degs_OLGs

up_degs_OLGs <- unique(c(up_degs_OLGs, additional_degs))

In [ ]:
# calculate average expression of set of marker genes across choir clusters
OLG_avg_deg_exp = grouped_obs_mean(olg[:, up_degs_OLGs], group_key='final_clusters', layer = 'log1p_norm')

In [ ]:
%%R -i OLG_avg_deg_exp

hmap_mat = t(scale(t(OLG_avg_deg_exp)))

In [ ]:
%%R
# create gene annotation highlighting select genes
selectGenes <- c('MBP', 'MOBP', 'MOG',
                                    'HSPA9', 'DNAJA4', "BAG3","SOD1","SORBS1", "XPO1",
                                    'CHRM5', 'TMPRSS5', 'ADAMTS18', "PEBP1", "SEMA3E", "ENO4",
                                    'MSMO1', 'FDFT1', 'LSS')

selectGenes <- selectGenes[order(match(selectGenes,rownames(hmap_mat)))]

ha = rowAnnotation(genes = anno_mark(at = which(rownames(hmap_mat) %in% selectGenes), 
    labels = selectGenes, labels_gp = gpar(col = "black", fontfamily = "Arial", 
                                           fontface="italic", fontsize = 10)))

In [ ]:
%%R

library(dendextend)

In [ ]:
%%R

head(hmap_mat)

In [ ]:
%%R -w 5 -h 8 -r 300 --units in
# plot heatmap
set.seed(123)

hmap = Heatmap(name = "Scaled Avg. Exp.", hmap_mat, col = circlize::colorRamp2(breaks = c(-1.5, -0.75, 0, 0.75, 1.5),
                                                                #colors=RColorBrewer::brewer.pal(5, "BuPu")),
                                                                colors=viridis::viridis_pal(option = "magma")(5)),
        right_annotation = ha,
        show_row_names = FALSE,
        cluster_rows = T,
        cluster_columns = as.dendrogram(hclust(dist(t(hmap_mat)))) %>% rotate(2:1),
        #row_names_gp = grid::gpar(fontsize = 2),
        heatmap_legend_param = list(direction = "horizontal",
                                   title_position = "topcenter"
                                   )
)

draw(hmap)

In [ ]:
# export DEGs lists for IPA pathway analysis

# OPC-A
temp = tt_OPCa[~(tt_OPCa.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OPCa_v_OPCcontrol_edgeR_results_IPAfilt.csv')

# OPC-B
temp = tt_OPCb[~(tt_OPCb.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OPCb_v_OPCcontrol_edgeR_results_IPAfilt.csv')

# OPC-C
temp = tt_OPCc[~(tt_OPCc.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OPCc_v_OPCcontrol_edgeR_results_IPAfilt.csv')


# OPC-D
temp = tt_OPCd[~(tt_OPCd.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OPCd_v_OPCcontrol_edgeR_results_IPAfilt.csv')



In [ ]:
# OLG-B
temp = tt_OLGb[~(tt_OLGb.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OLGb_v_OLGcontrol_edgeR_results_IPAfilt.csv')

# OLG-C
temp = tt_OLGc[~(tt_OLGc.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OLGc_v_OLGcontrol_edgeR_results_IPAfilt.csv')


# OLG-D
temp = tt_OLGd[~(tt_OLGd.index.str.startswith('MT-'))].copy()
temp = temp[~temp.index.str.startswith('RPL')].copy()
temp = temp[~temp.index.str.startswith('RPS')].copy()
temp.to_csv('../output/human/human_oligos_OLGd_v_OLGcontrol_edgeR_results_IPAfilt.csv')
